# <p align="center"> Identifying and Removing all the Industry data </p>

In [1]:
#Initialize Notebook Environment
from pathlib import Path
import sys
sys.path.insert( 0, Path("../..").resolve().absolute().__str__() )

In [2]:
from xaidar.filesUtils import loadPickle

lst_sessions = list( loadPickle("lst_sessions.pkl") )
print(f"Loaded {len(lst_sessions)} sessions from lst_sessions.pkl")
# Example of list of sessions
print(lst_sessions[0] )

Loaded 1007 sessions from lst_sessions.pkl
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/lb13385-88


In [3]:
industry_sessions = []
industry_labels = [ "sw", "in" ]
for session in lst_sessions:
    sesh_label = session.split("/")[-1][:2]
    if sesh_label in industry_labels:
        industry_sessions.append(session)

In [4]:
print(f"Filtered {len(industry_sessions)} industry sessions from {len(lst_sessions)} total sessions")
# Example of industry sessions
for sesh in industry_sessions:
    print(sesh)
# print(industry_sessions)

Filtered 65 industry sessions from 1007 total sessions
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1
data/2020/sw24758-9
data/2020/sw26558-8
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26591-1
data/2020/sw26591-3
data/2019/sw24814-5
data/2021/sw29506-1
data/2020/sw27229-11
data/2020/sw25092-13
data/2020/sw27206-1
data/2021/sw27230-11
data/2020/sw26591-1
data/2020/sw26557-2
data/2020/sw25092-11
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw20020-61
data/2020/sw26558-1
data/2021/sw29435-7
data/2020/sw26557-9
data/2018/lb19758-9/processing/analysis/TMP_pandda/sw25092-3
data/2018/lb19758-9/processing/analysis/TMP_pandda/sw26558-1
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/in22933-9
data/2020/sw27229-6
data/2021/sw28047-9
data/2020/sw26557-5
data/2019/sw25092-7
data/2018/lb19758-9/processing/analysis/TMP_pandda/sw26591-1
data/2020/sw26591-7
data/2020/sw24758-13
data/2020/sw25092-9
data/2021/sw29435-1


In [5]:
lst_treeObj_sessions = { "tree": [], "path": []}
for sesh in industry_sessions:
    seshPath = sesh.split("/")
    year, session = seshPath[1], seshPath[2]
    lst_treeObj_sessions[ "path" ].append( sesh )
    lst_treeObj_sessions[ "tree" ].append( f"tree_{year}_{session}.pkl")


In [13]:
from xaidar.treeObj import convertPathtoID , findAllFolderFiles
numb_sessions = len(lst_treeObj_sessions["tree"])
dic_filePaths = {}
print(f"Number of sessions with tree objects: {numb_sessions}")

for idx in range(numb_sessions):
    fileName = lst_treeObj_sessions[ "tree" ][idx]
    dirPath = lst_treeObj_sessions[ "path" ][idx]
    tree = loadPickle( Path(f"../../../data/treeObjs/xchem/data/{ fileName }" ))
    dirID = convertPathtoID( tree["fileTree"], tree["foldersCount"], dirPath)
    # print(f"Session {idx+1}/{numb_sessions}: {dirPath} -> ID: {dirID}")
    results = findAllFolderFiles( tree["fileTree"], tree["foldersCount"], dirID )
    dic_filePaths[ dirPath ] = results

Number of sessions with tree objects: 65


: 

In [10]:
from xaidar.treeObj import findAllFolderFiles
results = findAllFolderFiles( tree["fileTree"], tree["foldersCount"], dirID )

In [11]:
print(f"Found {len(results)} files in the directory {dirPath} with ID {dirID}")

Found 2905 files in the directory data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1 with ID [0, 0, 0, 1, 1, 1, 116]


In [12]:
for file in results[:5]:
    print(file)

data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/AGIOS1-x0266_normalised.pdb
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/log.txt
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/processed/-1.ccp4
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/processed/0.ccp4
data/2017/lb18145-17/processing/analysis/TMP_dataset_clustering/sw26558-1/processed/1.ccp4


In [8]:
from xaidar.treeObj import viewSubtree
folderID = dirID 
viewSubtree(tree["fileTree"], tree["foldersCount"], 1, len( folderID ) + 2, folderID=folderID )

data
└── 2017
    └── lb18145-17
        └── processing
            └── analysis
                └── TMP_dataset_clustering
                    └── sw26558-1
                        ├── [0] aligned
                        ├── [1] copied
                        ├── [2] processed
                        ├── [3-f] AGIOS1-x0266_normalised.pdb
                        └── [4-f] log.txt


In [15]:
from xaidar.treeObj import getGamma, pinchLevel, convertIDtoPath

def getAllSubFolders(tree, foldersCount,  lst_ParentIDs = None ,  lst_AllFolderIDs =  list[list[list]], folderPath=None):
    """
    Recursively gets all subfolders within a given folder.

    This function traverses a folder structure, starting from a given list of parent folder IDs, and
    collects all subfolders at each level of the hierarchy. It continues to call itself
    recursively until no more subfolders are found.

    Args:
        tree (object): An object representing the overall folder structure. The exact
            structure of this object is not specified by the function, but it's passed
            through recursive calls.
        foldersCount (list of lists): A nested list where `foldersCount[depth][parent_id]`
            indicates the number of subfolders for a given parent folder at a specific depth.
        lst_ParentIDs (list of list of folder IDs):A nested list that will be populated with
            the IDs of all subfolders found. Each index in this list corresponds to a
            different level of subfolders, where `lst_ParentIDs[level]` contains a list
            of all subfolder IDs at that level. The IDs are represented as lists of integers,
            where each integer corresponds to a specific subfolder in the parent folder. The 
            first level (0) corresponds to the targer folder of interest.
        lst_folderIDs (list, optional): A list of parent folder IDs to search for subfolders.
            Each ID is represented as a list of integers (e.g., `[ [0, 1] ]` for the second folder
            in the first subfolder of the root). Defaults to None, which implies the initial
            call should be handled appropriately by the calling function.
        folderPath (str, optional): A string representing the path to the folder. This
            parameter is currently not used in the function's logic. Defaults to None.
        lst_subfolderIDs (list, optional): A list to store the subfolder IDs found at each
            level. It's a nested list where `lst_subfolderIDs[level]` contains a list of
            all subfolder IDs at that level. Defaults to None, and is initialized within
            the function's logic if not provided.

    Returns:
        list: A nested list (`lst_subfolderIDs`) containing the IDs of all subfolders,
              grouped by their depth relative to the initial set of folders.

    Example:
        If you have a root folder with two subfolders, each having one subfolder of its own,
        the output might look like this:
        `[ [[0], [1]], [[0, 0], [1, 0]] ]`
        - The first sublist `[[0], [1]]` contains the IDs of the first level of subfolders.
        - The second sublist `[[0, 0], [1, 0]]` contains the IDs of the second level.
    """
    lst_subfolders_in_level = [ ]

    # Get a list of subfolderIDs for each folder ID
    for folderID in lst_ParentIDs:
        print(folderID)
 
        subFolderLevel = len(folderID) + 1
        folderGamma = getGamma(foldersCount, folderID)
        numberChildrenFolderS = foldersCount[subFolderLevel - 1][folderGamma]
        # print(f"Number of Children Folders: {numberChildrenFolderS}")
        if numberChildrenFolderS != 0:
            lst_subfolders_in_level.extend( [folderID + [id] for id in range(numberChildrenFolderS)] )

    # print(f"Length of lst subfolders in level: {len(lst_subfolders_in_level)}")

    # If no subfolder has been identified
    if len(lst_subfolders_in_level) == 0:
        # print(lst_subfolderIDs)
        return lst_AllFolderIDs
    else:
        lst_AllFolderIDs = lst_ParentIDs.append( lst_subfolders_in_level ) 
        new_parentFolderIDs = lst_subfolders_in_level
        # lst_subfolderIDs.append(lst_subfolders_in_level)
        # print(f"Lenght of output list: {len(lst_subfolderIDs)}")
        return getAllSubFolders(tree, foldersCount, lst_ParentIDs = new_parentFolderIDs, lst_AllFolderIDs=lst_AllFolderIDs)



def getFiles(tree,  foldersCount, folderID = None, folderPath = None):
    """
    Get all files in a folder based on folderID or folderPath.
    Args:
    - startDepth: Minimum is 1
    - folderID: Smallest is [ 0 ] -> root 
    Return: ( fileIDS, filePaths )
    - fileIDS (list): List of file IDs in the folder
    - filePaths (list): List of file paths in the folder
    """
    if folderPath != None:
        print( folderPath )
        folderID =  convertPathtoID( tree, foldersCount,  folderPath)

    fileIDS = []
    filesNameS = []

    fileLevel = len( folderID ) + 1 
    folderGamma = getGamma( foldersCount, folderID )
    directory = pinchLevel( tree,  fileLevel , flat = False)[folderGamma]
    
    numberChildrenFolderS = foldersCount[ fileLevel  - 1 ][folderGamma]
    numberFiles = len( directory[ numberChildrenFolderS: ] )     
    fileIDS.extend( [folderID + [ fileID ] for fileID in range( numberChildrenFolderS, numberChildrenFolderS + numberFiles ) ] )
    if folderPath != None:
        filesNameS.extend( pinchLevel( tree,  fileLevel , flat = False)[folderGamma][ numberChildrenFolderS: ] )   
        filePaths = [ folderPath + "/" + fileName for fileName in filesNameS ]

    if folderPath != None:
        return fileIDS, filePaths
    else:
        return fileIDS, [ convertIDtoPath(tree, foldersCount, fileID ) for fileID in fileIDS]

def findAllFolderFiles(folderID, tree, foldersCount):
    """
    Gets a list of object keys for all the objects that live within a specific folder.

    Args:
        folderID (list) : A list of indexes representing a folderID. I.e. [0,1,2] for 
        /root/folder2/subfolder3_in_folder2.
        tree (treeObj) : A list of nested lists representing the file 
    """
    allFoldersIDs = getAllSubFolders(tree, foldersCount, lst_ParentIDs =  list( folderID )    )
    flatFoldersIDs = allFoldersIDs[:1] + [ item for lst in allFoldersIDs[1:] for item in lst]
    allPaths = [  getFiles( tree, foldersCount, folderID=folderID )[1] for folderID in flatFoldersIDs]
    flatPaths = [ item for lst in allPaths for item in lst]
    return flatPaths

In [16]:
# from xaidar.treeObj import findAllFolderFiles
lst_objKeys = {}
numb_sessions = len( lst_treeObj_sessions[ "tree" ] )
for idx in [0]: # range( numb_sessions):
    fileName = lst_treeObj_sessions[ "tree" ][idx]
    tree = loadPickle( Path(f"../../../data/treeObjs/xchem/data/{ fileName }" ))
    dirPath =  lst_treeObj_sessions[ "path" ][idx] 
    dirID = convertPathtoID( tree["fileTree"], tree["foldersCount"], dirPath)
    print(dirID)
    # lst_objKeys[dirPath] = findAllFolderFiles( tree["fileTree"], tree["foldersCount"], dirID)


[0, 0, 0]


In [17]:
print(dirID)

[0, 0, 0]


In [ ]:

def getAllSubFolders(tree, foldersCount, lst_folderIDs=None,  lst_AllFolderIDs = []):
    """
    Recursively gets all subfolders within a given folder.

    This function traverses a folder structure, starting from a given list of parent folder IDs, and
    collects all subfolders at each level of the hierarchy. It continues to call itself
    recursively until no more subfolders are found.

    Args:
        tree (object): An object representing the overall folder structure. The exact
            structure of this object is not specified by the function, but it's passed
            through recursive calls.
        foldersCount (list of lists): A nested list where `foldersCount[depth][parent_id]`
            indicates the number of subfolders for a given parent folder at a specific depth.
        lst_folderIDs (list, mandatory): A list of parent folder IDs to search for subfolders.
            Each ID is represented as a list of integers (e.g., `[0, 1]` for the second folder
            in the first subfolder of the root). Defaults to None, which implies the initial
            call should be handled appropriately by the calling function.
        lst_subfolderIDs (list, don't use): Internal argument of recursive function.

    Returns:
        list: A nested list (`lst_subfolderIDs`) containing the IDs of all subfolders,
              grouped by their depth relative to the initial set of folders.

    Example:
        If you have a root folder with two subfolders, each having one subfolder of its own,
        the output might look like this:
        `[ [[0], [1]], [[0, 0], [1, 0]] ]`
        - The first sublist `[[0], [1]]` contains the IDs of the first level of subfolders.
        - The second sublist `[[0, 0], [1, 0]]` contains the IDs of the second level.
    """
    print( f"Parent folder IDs: {lst_folderIDs}" )
    print( f"Output list: {lst_AllFolderIDs}" )
    lst_subfolders_in_level = []
    new_parentFolderIDs = []

    # Get a list of subfolderIDs for each folder ID
    for folderID in lst_folderIDs:
        # folderID = [] # placeholder

        subFolderLevel = len(folderID) + 1
        folderGamma = getGamma(foldersCount, folderID)
        numberChildrenFolderS = foldersCount[subFolderLevel - 1][folderGamma]
        print(f"Number of Children Folders: {numberChildrenFolderS}")
        if numberChildrenFolderS != 0:
            lst_subfolders_in_level.extend([folderID + [id] for id in range(numberChildrenFolderS)])

    print(f"Length of lst subfolders in level: {len(lst_subfolders_in_level)}")
    print(f"Subfolders in level: {lst_subfolders_in_level}")
    print("-------")
    # If no subfolder has been identified
    if len(lst_subfolders_in_level) == 0:
        lst_AllFolderIDs.append( lst_folderIDs )
        return lst_AllFolderIDs
    else:
        
        lst_AllFolderIDs.append( lst_folderIDs )
        new_parentFolderIDs = lst_subfolders_in_level
        return getAllSubFolders(tree, foldersCount, lst_folderIDs = new_parentFolderIDs, lst_AllFolderIDs = lst_AllFolderIDs)



In [55]:
allFoldersIDs = getAllSubFolders( tree["fileTree"], tree["foldersCount"], lst_folderIDs =  [ dirID ]   )
# allFoldersIDs = getAllSubFolders( tree["fileTree"], tree["foldersCount"], lst_ParentIDs =  [ dirID ]   )
print(allFoldersIDs)
# flatFoldersIDs = allFoldersIDs[:1] + [ item for lst in allFoldersIDs[1:] for item in lst]

Parent folder IDs: [[0, 0, 0]]
Output list: []
Number of Children Folders: 1
Length of lst subfolders in level: 1
Subfolders in level: [[0, 0, 0, 0]]
-------
Parent folder IDs: [[0, 0, 0, 0]]
Output list: [[[0, 0, 0]]]
Number of Children Folders: 0
Length of lst subfolders in level: 0
Subfolders in level: []
-------
[[[0, 0, 0]], [[0, 0, 0, 0]]]


In [18]:
lst_objKeys[dirPath] = findAllFolderFiles( tree["fileTree"], tree["foldersCount"], dirID)

['data']


TypeError: 'int' object is not subscriptable

In [ ]:
print( allFolders )

[[0, 0, 0, 1, 1, 2, 110], [[0, 0, 0, 1, 1, 2, 110, 0], [0, 0, 0, 1, 1, 2, 110, 1], [0, 0, 0, 1, 1, 2, 110, 2]], [[0, 0, 0, 1, 1, 2, 110, 1, 0], [0, 0, 0, 1, 1, 2, 110, 1, 1], [0, 0, 0, 1, 1, 2, 110, 1, 2], [0, 0, 0, 1, 1, 2, 110, 1, 3], [0, 0, 0, 1, 1, 2, 110, 1, 4], [0, 0, 0, 1, 1, 2, 110, 1, 5], [0, 0, 0, 1, 1, 2, 110, 1, 6], [0, 0, 0, 1, 1, 2, 110, 1, 7], [0, 0, 0, 1, 1, 2, 110, 1, 8], [0, 0, 0, 1, 1, 2, 110, 1, 9], [0, 0, 0, 1, 1, 2, 110, 1, 10], [0, 0, 0, 1, 1, 2, 110, 1, 11], [0, 0, 0, 1, 1, 2, 110, 1, 12], [0, 0, 0, 1, 1, 2, 110, 1, 13], [0, 0, 0, 1, 1, 2, 110, 1, 14], [0, 0, 0, 1, 1, 2, 110, 1, 15], [0, 0, 0, 1, 1, 2, 110, 1, 16], [0, 0, 0, 1, 1, 2, 110, 1, 17], [0, 0, 0, 1, 1, 2, 110, 1, 18], [0, 0, 0, 1, 1, 2, 110, 1, 19], [0, 0, 0, 1, 1, 2, 110, 1, 20], [0, 0, 0, 1, 1, 2, 110, 1, 21], [0, 0, 0, 1, 1, 2, 110, 1, 22], [0, 0, 0, 1, 1, 2, 110, 1, 23], [0, 0, 0, 1, 1, 2, 110, 1, 24], [0, 0, 0, 1, 1, 2, 110, 1, 25], [0, 0, 0, 1, 1, 2, 110, 1, 26], [0, 0, 0, 1, 1, 2, 110, 1, 27], [